# Week 9: Text processing, pandas, and the capstone repository

**Month 3 starts your capstone: Experiment A, biomedical literature retrieval.** From now on, the code you write is **portfolio code**. It lives in its own repository, `capstone/`, which you'll eventually make public on GitHub as `medsearch-lab`.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Tokenisation, stop words, stemming (NLTK) | 2 h |
| Day 2 | pandas for data exploration: NFCorpus | 2.5 h |
| Day 3 | A scalable `InvertedIndex` class, test-first | 3 h |
| Day 4 | `DATASETS.md`, profiling, repo hygiene | 2 h |
| Review | Spaced recall | 0.5 h |

**Read:** *Introduction to Information Retrieval*, chapter 2 (sections 2.1–2.2) and chapter 1.

### Capstone milestone M1 (end of this week)
- [ ] `capstone/` is its own Git repo with `medlab/text.py`, `medlab/index.py` and tests.
- [ ] The index builds over all of NFCorpus in under 60 seconds, and saves/loads from disk.
- [ ] `capstone/DATASETS.md` documents NFCorpus: source, licence, size, splits, how relevance was judged, and limitations.

### One-time setup
```bash
cd ~/ai-programme
source .venv/bin/activate
pip install -r requirements.txt        # now includes nltk, pyyaml, rank_bm25
mkdir -p capstone/medlab capstone/tests && touch capstone/medlab/__init__.py
echo "capstone/" >> .gitignore          # the capstone gets its own repo
cd capstone && git init -b main
```
The first time you load NFCorpus, it downloads about 3 MB into `ai-programme/data/`.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

---
# Day 1: From text to tokens (2 h)

Search engines and classifiers never see "text". They see **tokens**, and preprocessing choices change results more than people expect:

| Step | Example | Effect |
|---|---|---|
| Lowercase | "Vitamin" → "vitamin" | merges variants |
| Tokenise | "COVID-19's" → covid, 19, s | defines what a "word" is |
| Stop words | remove "the", "of", "and" | smaller index; less noise in ranking |
| Stemming | "cancers", "cancerous" → "cancer" | matches word variants (recall ↑), sometimes wrongly (precision ↓) |

In [ ]:
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

tokenizer = RegexpTokenizer(r"[a-z0-9]+")     # runs of letters or digits
stemmer = PorterStemmer()
text = "Dietary fibres reduce the risks of colorectal cancers in older adults."
tokens = tokenizer.tokenize(text.lower())
print(tokens)
print([t for t in tokens if t not in ENGLISH_STOP_WORDS])
print([stemmer.stem(t) for t in tokens])
print("stemming quirks:", [stemmer.stem(w) for w in ["university", "universe", "organ", "organization", "news"]])

### Exercise 9.1: `preprocess` in your capstone package
Create **`capstone/medlab/text.py`** in VS Code with:
```python
def preprocess(text, stem=True, remove_stop=True) -> list[str]:
```
It should lowercase the text, tokenise with the regex `[a-z0-9]+`, optionally drop scikit-learn's `ENGLISH_STOP_WORDS`, and optionally apply the Porter stemmer. **Performance tip:** stemming is slow, so cache it in a module-level dict `{token: stem}`. You'll process about a million tokens.

In [ ]:
# Create capstone/medlab/text.py in VS Code, then run the checker.

In [ ]:
text_mod = fresh("medlab.text")
pp = text_mod.preprocess
assert pp("The Cancers of the LUNG!") == ["cancer", "lung"]
assert pp("The Cancers", stem=False) == ["cancers"]
assert pp("the of and", remove_stop=True) == []
assert pp("Vitamin-D 2024", stem=False, remove_stop=False) == ["vitamin", "d", "2024"]
print("✅ 9.1 passed")

---
# Day 2: pandas for exploration (2.5 h)

**pandas** is the spreadsheet of Python. You'll use it in almost every data job. The core ideas:
- a **DataFrame** is a table: named columns, one row per record,
- **vectorised column operations** (`df["n"] * 2`), no loops,
- **filter** with boolean masks: `df[df["n"] > 100]`,
- **group and aggregate**: `df.groupby("col")["x"].mean()`,
- `describe()`, `value_counts()`, `hist()` for a first look.

**Always look at your data before modelling it.** Most bugs in AI projects are data bugs.

In [ ]:
import pandas as pd
from course.data import load_nfcorpus

corpus, queries, qrels = load_nfcorpus("test")
docs = pd.DataFrame([{"doc_id": d, "title": v["title"], "text": v["text"]} for d, v in corpus.items()])
docs["n_tokens"] = docs["text"].str.split().str.len()
print(docs.shape)
docs.head()

In [ ]:
print(docs["n_tokens"].describe())
docs["n_tokens"].hist(bins=50); import matplotlib.pyplot as plt; plt.xlabel("tokens per document"); plt.show()
print("A random query:", next(iter(queries.items())))

### Exercise 9.2: Describe the benchmark
Build a DataFrame `qrels_df` with columns `query_id, doc_id, relevance` (one row per judgement). Then fill in the `stats` dict. Use pandas operations, not loops, wherever you can.

In [ ]:
qrels_df = None
stats = {
    "n_docs": None,                    # number of documents in the corpus
    "n_queries": None,                 # number of test queries
    "mean_doc_tokens": None,           # mean whitespace tokens per document
    "n_judgements": None,              # rows in qrels_df
    "mean_relevant_per_query": None,   # mean number of judged-relevant docs per query
    "share_grade_2": None,             # fraction of judgements with relevance == 2
}

In [ ]:
assert list(qrels_df.columns) == ["query_id", "doc_id", "relevance"]
assert stats["n_docs"] == len(corpus) and stats["n_queries"] == len(queries)
assert stats["n_judgements"] == sum(len(v) for v in qrels.values())
assert abs(stats["mean_relevant_per_query"] - stats["n_judgements"] / stats["n_queries"]) < 1e-9
assert 0 <= stats["share_grade_2"] <= 1
for k, v in stats.items(): print(f"{k:25s} {v:.2f}" if isinstance(v, float) else f"{k:25s} {v}")
print("✅ 9.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`pd.DataFrame([{"query_id": q, "doc_id": d, "relevance": r} for q, rels in qrels.items() for d, r in rels.items()])`

</details>

<details><summary>Hint 2</summary>

`qrels_df.groupby("query_id").size().mean()` and `(qrels_df["relevance"] == 2).mean()`.

</details>


✍️ *What surprised you about the data? (Document lengths? The number of relevant documents per query? What the queries look like?) Write 3 observations. They'll go into DATASETS.md.* (double-click to edit)



---
# Day 3: An `InvertedIndex` that scales (3 h)

Week 4's index stored *which* documents contain a term. Ranking (next week) also needs **how often** a term occurs in each document and **how long** each document is:
```
postings: {term: {doc_id: term_frequency}}
doc_len:  {doc_id: number_of_tokens_after_preprocessing}
```
### Exercise 9.3: Write `capstone/medlab/index.py`, test-first
Write `capstone/tests/test_index.py` **first** (at least 6 tests, on a tiny hand-made corpus where you know the answers). Then implement:
```python
class InvertedIndex:
    postings: dict[str, dict[str, int]]
    doc_len: dict[str, int]

    @classmethod
    def build(cls, docs: dict[str, str], analyzer) -> "InvertedIndex"   # analyzer: text -> tokens
    n_docs  (property)        avgdl (property: mean doc length)
    def df(self, term) -> int                                            # document frequency
    def save(self, path) / @classmethod load(cls, path)                  # use pickle
```
`collections.Counter(tokens)` gives term frequencies in one line. Pytest's built-in `tmp_path` fixture gives you a temporary folder for the save/load test.

In [ ]:
# Write the tests and the class in VS Code, then run the checker.

In [ ]:
import time
assert run_tests("tests/test_index.py") == 0
assert (CAP / "tests/test_index.py").read_text().count("def test_") >= 6
index_mod = fresh("medlab.index"); text_mod = fresh("medlab.text")
doc_texts = {d: v["title"] + " " + v["text"] for d, v in corpus.items()}
t0 = time.perf_counter()
index = index_mod.InvertedIndex.build(doc_texts, text_mod.preprocess)
build_s = time.perf_counter() - t0
print(f"{index.n_docs} docs, {len(index.postings)} terms, avgdl {index.avgdl:.1f}, built in {build_s:.1f}s")
assert index.n_docs == len(corpus) and build_s < 60
some_term = text_mod.preprocess("cancer")[0] if text_mod.preprocess("cancer") else next(iter(index.postings))
assert index.df(some_term) == len(index.postings.get(some_term, {}))
(Path("../data")).mkdir(exist_ok=True)
index.save("../data/nfcorpus_index.pkl")
assert index_mod.InvertedIndex.load("../data/nfcorpus_index.pkl").doc_len == index.doc_len
print("✅ 9.3 passed (milestone M1, part 1)")

---
# Day 4: Documentation, profiling, repo hygiene (2 h)

### `DATASETS.md`
Professional ML projects document their data. Create `capstone/DATASETS.md` with an NFCorpus section covering:
- **Source and citation:** Boteva et al. (2016), *A Full-Text Learning to Rank Dataset for Medical Information Retrieval* (ECIR), obtained through the BEIR benchmark (Thakur et al., 2021).
- **Licence / terms of use:** look it up on the original NFCorpus page and the BEIR repository, and quote it exactly. If it's unclear, say so and restrict yourself to non-commercial research use.
- **Contents:** your `stats` from Day 2.
- **How the relevance labels were made**, and what that implies (hint: they come from links on NutritionFacts.org, not from expert relevance judgements of every document).
- **Known limitations:** for example, incomplete judgements (unjudged ≠ irrelevant), the domain (nutrition-heavy), English only.
- **No patient data:** state explicitly that the corpus contains published abstracts only.

### Profiling: where does the time go?
Never guess what's slow. Measure it:

In [ ]:
import cProfile, pstats
sample = dict(list(doc_texts.items())[:500])
cProfile.run("index_mod.InvertedIndex.build(sample, text_mod.preprocess)", "../data/build.prof")
pstats.Stats("../data/build.prof").sort_stats("cumulative").print_stats(8)

✍️ *Which function takes the most time? If you hadn't cached the stemmer, what would this profile look like? Try it: temporarily remove the cache and compare the build time.* (double-click to edit)



### Repo hygiene checklist for `capstone/`
- [ ] `README.md`: one paragraph on what the project is, plus the **not clinically validated** disclaimer.
- [ ] `.gitignore`: `data/`, `__pycache__/`, `.pytest_cache/`, `*.pkl`, `results/raw/`.
- [ ] `requirements.txt` for the capstone.
- [ ] Your commits: `Add text preprocessing with tests`, `Add inverted index with save/load`, `Document NFCorpus in DATASETS.md`.
- [ ] Create a **private** GitHub repo `medsearch-lab` and push. You'll make it public in Week 22.

---
# Review (30 min)
1. *(Week 6)* What's the stability condition for the gradient-descent learning rate?
2. *(Week 7)* What is MAP in one sentence? How does it relate to L2?
3. *(Week 8)* Name three types of leakage and the fix for each.
4. What does stemming trade off?
5. What is document frequency, and why will it matter next week?
6. What's the difference between `df.groupby("q").size()` and `df["q"].value_counts()`?

✍️ *Your answer:* (double-click to edit)

